# Link-Kind Audit

Classifies every NCIt and LOINC link in the `cdisc-for-ai` reference files by
**link kind** and **provenance class**, and counts the actual rows behind each.

Purpose: decide one predicate per link kind before rendering DSS/BC content as
RDF in `cosmos-rdf` and `usdm-rdf`. A single `skos:exactMatch` currently hides
at least five different statements.

**Link kinds**

| Kind | Statement |
|---|---|
| `identity` | this thing *is* that NCIt concept (the C-code is its identifier) |
| `membership` | this concept is a permissible value in that codelist |
| `binding` | this variable draws its values from that codelist |
| `denotation` | this model element *means* that concept |
| `pinned_value` | this variable is fixed to that value in this specialization |
| `value_list` | this variable is restricted to these values in this specialization |
| `category_tag` | this BC carries that category token - the object end is a label, not an identifier |
| `mapping` | this concept corresponds to a concept in another terminology |

**Provenance classes**

| Class | Meaning |
|---|---|
| `published` | CDISC / NCI EVS asserts it in a release |
| `derived` | computed by a rule from published inputs |
| `curated` | a judgement, needs a justification |

Counts are measured from the workbooks. The kind/class assignment in
`INVENTORY` is authored in this notebook and is the part to review.

In [1]:
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent.parent  # notebooks/ -> link-semantics/ -> repo root
OUT = REPO / "link-semantics" / "interim" / "Link_Kind_Audit.xlsx"

# Optional: path to a previous Link_Kind_Audit.xlsx for the version-bump diff in the
# final cell. Leave as None to skip.
PRIOR_FILE = None

GRAPH = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph.xlsx"
GRAPH_CT = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph_CT.xlsx"
TEST_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Test_Identity.xlsx"
INSTRUMENT_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Instrument_Identity.xlsx"
CROSS_REFS = REPO / "sdtm-test-codes" / "interim" / "Codelist_Cross_References.xlsx"

for path in [GRAPH, GRAPH_CT, TEST_IDENTITY, INSTRUMENT_IDENTITY, CROSS_REFS]:
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - run this notebook from link-semantics/notebooks/")

print("repo:", REPO)

repo: /sessions/rcw-011ndgbpomxbp7qzdpv1ucjv/mnt/cdisc-for-ai


In [2]:
SHEETS = {}


def sheet(path, name):
    key = (path.name, name)
    if key not in SHEETS:
        SHEETS[key] = pd.read_excel(path, sheet_name=name)
    return SHEETS[key]

## Pins

What this run was executed against. The SDTM CT release date is not recorded
in any of the input files (the extract reads a versionless NCI EVS URL), so it
cannot be pinned from data here.

In [3]:
import subprocess


def generated(path):
    book = pd.ExcelFile(path)
    readme_name = [name for name in book.sheet_names if name.lower() == "readme"][0]
    readme = pd.read_excel(book, sheet_name=readme_name, header=None)[0].dropna().astype(str)
    return readme[readme.str.startswith("Generated")].iloc[0].split(":", 1)[1].strip()


commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
package_dates = sheet(GRAPH, "DSS")["package_date"]

pins = pd.DataFrame(
    [{"pin": "cdisc-for-ai commit", "value": commit}]
    + [{"pin": "COSMoS package_date (latest in DSS sheet)", "value": str(package_dates.max())}]
    + [{"pin": "COSMoS package_date (distinct values)", "value": str(package_dates.nunique())}]
    + [
        {"pin": f"{path.name} generated", "value": generated(path)}
        for path in [GRAPH, GRAPH_CT, TEST_IDENTITY, INSTRUMENT_IDENTITY, CROSS_REFS]
    ]
)
pins

,pin,value
0,cdisc-for-ai commit,c3549da46c92a9ff191f6ccb46706922955bebfd
1,COSMoS package_date (latest in DSS sheet),2026-07-14
2,COSMoS package_date (distinct values),7
3,COSMoS_Graph.xlsx generated,2026-08-23
4,COSMoS_Graph_CT.xlsx generated,2026-09-30
5,SDTM_Test_Identity.xlsx generated,2026-10-01 10:39:19
6,SDTM_Instrument_Identity.xlsx generated,2026-10-01 10:39:28
7,Codelist_Cross_References.xlsx generated,2026-10-01 10:39


## Inventory

One row per place where a link is stored. `subject` and `object` name the
columns that carry the two ends. `filter` is a pandas query applied first.
`justification` names a column that already records *how* the link was
established, or `None` where none exists.

In [4]:
INVENTORY = [
    {
        "link_kind": "identity",
        "statement": "BC is an NCIt concept",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "BC",
        "subject": ["bc_id"],
        "object": "ncit_code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "DEC is an NCIt concept",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "DataElementConcepts",
        "subject": ["dec_id"],
        "object": "ncit_dec_code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "TESTCD is an NCIt concept",
        "provenance": "published",
        "path": TEST_IDENTITY,
        "sheet": "Test Codes",
        "subject": ["TESTCD"],
        "object": "NCIt_Code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "codelist is an NCIt concept",
        "provenance": "published",
        "path": GRAPH_CT,
        "sheet": "Codelists",
        "subject": ["codelist_submission_value"],
        "object": "codelist_concept_id",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "membership",
        "statement": "concept is a permissible value in codelist",
        "provenance": "published",
        "path": GRAPH_CT,
        "sheet": "CodelistTerms",
        "subject": ["codelist_concept_id"],
        "object": "term_concept_id",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "binding",
        "statement": "variable draws values from codelist",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "codelist_concept_id",
        "filter": "codelist_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "denotation",
        "statement": "variable means the DEC",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "dec_id",
        "filter": "dec_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "pinned_value",
        "statement": "variable is fixed to an NCIt-coded value in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "assigned_term_concept_id",
        "filter": "assigned_term_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "pinned_value",
        "statement": "DSS pins an external LOINC code on a --LOINC variable",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "assigned_term_value",
        "filter": "variable_name.str.endswith('LOINC') and assigned_term_value.notna()",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "variable is restricted to a list of values from its bound codelist in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "value_list.notna() and codelist_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "variable is restricted to a list of values with no bound codelist in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "value_list.notna() and codelist_concept_id.isna() and not variable_name.str.endswith('LOINC')",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "DSS lists several external LOINC codes on a --LOINC variable",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "variable_name.str.endswith('LOINC') and value_list.notna()",
        "justification": None,
    },
    {
        "link_kind": "category_tag",
        "statement": "BC carries a category token (label, no identifier)",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "BC_Categories",
        "subject": ["bc_id"],
        "object": "category",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "BC corresponds to an external code (Coding sheet)",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Coding",
        "subject": ["bc_id"],
        "object": "code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "instrument codelist corresponds to an NCIt instrument concept (C20993)",
        "provenance": "curated",
        "path": INSTRUMENT_IDENTITY,
        "sheet": "Instruments",
        "subject": ["Codelist_TESTCD"],
        "object": "Instrument_NCIt_Code",
        "filter": "Instrument_NCIt_Code.notna()",
        "justification": "Instrument_Match_Method",
    },
    {
        "link_kind": "mapping",
        "statement": "instrument codelist corresponds to an NCIt question container (C211913)",
        "provenance": "derived",
        "path": INSTRUMENT_IDENTITY,
        "sheet": "Instruments",
        "subject": ["Codelist_TESTCD"],
        "object": "Container_NCIt_Code",
        "filter": "Container_NCIt_Code.notna()",
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "codelist pair declared semantically related",
        "provenance": "curated",
        "path": CROSS_REFS,
        "sheet": "Cross_Reference_Pairs",
        "subject": ["codelist_A_concept_id"],
        "object": "codelist_B_concept_id",
        "filter": None,
        "justification": "rationale",
    },
    {
        "link_kind": "membership",
        "statement": "term overlap/gap between a declared codelist pair",
        "provenance": "derived",
        "path": CROSS_REFS,
        "sheet": "Term_Diff",
        "subject": ["pair_id", "term_concept_id"],
        "object": "gap_classification",
        "filter": None,
        "justification": None,
    },
]

print(len(INVENTORY), "inventory entries")

18 inventory entries


In [5]:
def measure(entry):
    df = sheet(entry["path"], entry["sheet"])
    if entry["filter"] is not None:
        df = df.query(entry["filter"], engine="python")
    subject_cols = entry["subject"]
    object_col = entry["object"]
    rows = len(df)
    distinct_subjects = len(df[subject_cols].drop_duplicates())
    distinct_objects = df[object_col].nunique(dropna=True)
    missing_object = int(df[object_col].isna().sum())
    return {
        "link_kind": entry["link_kind"],
        "provenance": entry["provenance"],
        "statement": entry["statement"],
        "file": entry["path"].name,
        "sheet": entry["sheet"],
        "subject_cols": " + ".join(subject_cols),
        "object_col": object_col,
        "rows": rows,
        "distinct_subjects": distinct_subjects,
        "distinct_objects": distinct_objects,
        "object_missing": missing_object,
        "justification_col": entry["justification"] or "",
    }


audit = pd.DataFrame([measure(entry) for entry in INVENTORY])
audit

,link_kind,provenance,statement,file,sheet,subject_cols,object_col,rows,distinct_subjects,distinct_objects,object_missing,justification_col
0,identity,published,BC is an NCIt concept,COSMoS_Graph.xlsx,BC,bc_id,ncit_code,1475,1475,1469,6,
1,identity,published,DEC is an NCIt concept,COSMoS_Graph.xlsx,DataElementConcepts,dec_id,ncit_dec_code,6029,226,224,2,
2,identity,published,TESTCD is an NCIt concept,SDTM_Test_Identity.xlsx,Test Codes,TESTCD,NCIt_Code,5988,5969,5988,0,
3,identity,published,codelist is an NCIt concept,COSMoS_Graph_CT.xlsx,Codelists,codelist_submission_value,codelist_concept_id,297,297,297,0,
4,membership,published,concept is a permissible value in codelist,COSMoS_Graph_CT.xlsx,CodelistTerms,codelist_concept_id,term_concept_id,17742,295,12833,0,
5,binding,published,variable draws values from codelist,COSMoS_Graph.xlsx,Variables,ds_id + variable_name,codelist_concept_id,7438,7438,297,0,
6,denotation,published,variable means the DEC,COSMoS_Graph.xlsx,Variables,ds_id + variable_name,dec_id,10791,10791,199,0,
7,pinned_value,published,variable is fixed to an NCIt-coded value in th...,COSMoS_Graph.xlsx,Variables,ds_id + variable_name,assigned_term_concept_id,4300,4300,1307,0,
8,pinned_value,published,DSS pins an external LOINC code on a --LOINC v...,COSMoS_Graph.xlsx,Variables,ds_id + variable_name,assigned_term_value,98,98,98,0,
9,value_list,published,variable is restricted to a list of values fro...,COSMoS_Graph.xlsx,Variables,ds_id + variable_name,value_list,2310,2310,247,0,


## Summary by link kind and provenance

In [6]:
summary = (
    audit.groupby(["link_kind", "provenance"], as_index=False)
    .agg(entries=("rows", "size"), rows=("rows", "sum"))
    .sort_values(["link_kind", "provenance"])
)
summary

,link_kind,provenance,entries,rows
0,binding,published,1,7438
1,category_tag,published,1,4382
2,denotation,published,1,10791
3,identity,published,4,13789
4,mapping,curated,2,261
5,mapping,derived,1,359
6,mapping,published,1,105
7,membership,derived,1,676
8,membership,published,1,17742
9,pinned_value,published,2,4398


## Detail on the curated and mapping cases

These are the links that need a justification. Everything else is an authority
assertion carried by a release.

In [7]:
instruments = sheet(INSTRUMENT_IDENTITY, "Instruments")
match_methods = instruments["Instrument_Match_Method"].value_counts(dropna=False).rename_axis("method").reset_index(name="codelists")
match_methods

,method,codelists
0,exact,206
1,UNMATCHED,103
2,suffix-strip,44
3,normalized,10


In [8]:
coding = sheet(GRAPH, "Coding")
external_systems = (
    coding.groupby(["system_name", "system"], as_index=False)
    .agg(rows=("code", "size"), distinct_codes=("code", "nunique"), distinct_bcs=("bc_id", "nunique"))
)
external_systems

,system_name,system,rows,distinct_codes,distinct_bcs
0,LOINC,http://loinc.org/,105,98,105


### LOINC at two grains

BC-level (`Coding`) versus DSS-level (a `--LOINC` variable with a pinned
value). The question for RDF: the BC-level link is a mapping statement, the
DSS-level one is a value on a variable in a fixed context.

In [9]:
variables = sheet(GRAPH, "Variables")
bc_loinc = coding[coding["system_name"] == "LOINC"][["bc_id", "code"]].rename(columns={"code": "bc_level_loinc"})
loinc_vars = variables[variables["variable_name"].str.endswith("LOINC", na=False)]
assigned_loinc = loinc_vars[loinc_vars["assigned_term_value"].notna()][
    ["ds_id", "bc_id", "assigned_term_value"]
].rename(columns={"assigned_term_value": "dss_level_loinc"})
listed_loinc = loinc_vars[loinc_vars["value_list"].notna()][["ds_id", "bc_id", "value_list"]].copy()
listed_loinc["dss_level_loinc"] = listed_loinc["value_list"].str.split(";")
listed_loinc = listed_loinc.explode("dss_level_loinc")[["ds_id", "bc_id", "dss_level_loinc"]]
listed_loinc["dss_level_loinc"] = listed_loinc["dss_level_loinc"].str.strip()
dss_loinc = pd.concat([assigned_loinc, listed_loinc], ignore_index=True)

both = dss_loinc.merge(bc_loinc, on="bc_id", how="left")
fanout = dss_loinc.groupby("bc_id")["dss_level_loinc"].nunique()

grain = pd.DataFrame(
    [
        {"measure": "BCs with a BC-level LOINC", "count": bc_loinc["bc_id"].nunique()},
        {"measure": "DSSs pinning a LOINC value", "count": dss_loinc["ds_id"].nunique()},
        {"measure": "--LOINC variables", "count": len(loinc_vars)},
        {"measure": "... with one assigned value", "count": assigned_loinc["ds_id"].nunique()},
        {"measure": "... with a value list", "count": listed_loinc["ds_id"].nunique()},
        {"measure": "... with no value", "count": int((loinc_vars["assigned_term_value"].isna() & loinc_vars["value_list"].isna()).sum())},
        {"measure": "DSS x LOINC pairs", "count": len(dss_loinc)},
        {"measure": "distinct LOINC codes at DSS level", "count": dss_loinc["dss_level_loinc"].nunique()},
        {"measure": "pinned DSSs whose BC also has a BC-level LOINC", "count": both.loc[both["bc_level_loinc"].notna(), "ds_id"].nunique()},
        {"measure": "of those, BC-level code is among the DSS codes", "count": both.loc[both["bc_level_loinc"] == both["dss_level_loinc"], "ds_id"].nunique()},
        {"measure": "BCs whose DSSs pin more than one distinct LOINC", "count": int((fanout > 1).sum())},
    ]
)
grain

,measure,count
0,BCs with a BC-level LOINC,105
1,DSSs pinning a LOINC value,140
2,--LOINC variables,151
3,... with one assigned value,98
4,... with a value list,42
5,... with no value,11
6,DSS x LOINC pairs,183
7,distinct LOINC codes at DSS level,180
8,pinned DSSs whose BC also has a BC-level LOINC,11
9,"of those, BC-level code is among the DSS codes",11


### Value lists

`value_list` is a `;`-separated string on the variable. The audit table above
counts variables; this counts the values behind them.

In [10]:
listed = variables[variables["value_list"].notna()].copy()
both_set = listed["assigned_term_concept_id"].notna() | listed["assigned_term_value"].notna()
if both_set.any():
    raise ValueError(f"{int(both_set.sum())} variables carry both a value list and an assigned value")

listed["scope"] = "no codelist"
listed.loc[listed["codelist_concept_id"].notna(), "scope"] = "bound codelist"
listed.loc[listed["variable_name"].str.endswith("LOINC"), "scope"] = "LOINC"
listed["value"] = listed["value_list"].str.split(";")
exploded = listed.explode("value")
exploded["value"] = exploded["value"].str.strip()

value_list_detail = pd.DataFrame({
    "variables": listed.groupby("scope").size(),
    "dss": listed.groupby("scope")["ds_id"].nunique(),
    "with_subset_codelist": listed.groupby("scope")["subset_codelist"].count(),
    "values_total": exploded.groupby("scope").size(),
    "distinct_values": exploded.groupby("scope")["value"].nunique(),
}).reset_index()
value_list_detail

,scope,variables,dss,with_subset_codelist,values_total,distinct_values
0,LOINC,42,42,0,85,83
1,bound codelist,2310,896,296,6686,674
2,no codelist,648,241,0,5016,588


### `subset_codelist` naming

A value list on a codelist-bound variable may carry a `subset_codelist` name
(e.g. `NY_NY`). A *combination* is one bound codelist with one `value_list`
string, as published. How consistently is the same combination named?

In [11]:
bound_lists = listed[listed["scope"] == "bound codelist"].copy()
bound_lists["named"] = bound_lists["subset_codelist"].notna()

combinations = (
    bound_lists.groupby(["codelist_concept_id", "value_list"], as_index=False)
    .agg(variables=("named", "size"), named=("named", "sum"), names=("subset_codelist", "nunique"))
)
combinations["naming"] = "never"
combinations.loc[combinations["named"] > 0, "naming"] = "partly"
combinations.loc[combinations["named"] == combinations["variables"], "naming"] = "always"

named_rows = bound_lists[bound_lists["named"]]
combinations_per_name = (
    named_rows.drop_duplicates(["subset_codelist", "codelist_concept_id", "value_list"])
    .groupby("subset_codelist")
    .size()
)

subset_naming = pd.DataFrame(
    [
        {"measure": "codelist-bound value lists", "count": len(bound_lists)},
        {"measure": "... with a subset_codelist name", "count": int(bound_lists["named"].sum())},
        {"measure": "combinations (codelist, value_list)", "count": len(combinations)},
        {"measure": "... always named", "count": int((combinations["naming"] == "always").sum())},
        {"measure": "... partly named", "count": int((combinations["naming"] == "partly").sum())},
        {"measure": "... never named", "count": int((combinations["naming"] == "never").sum())},
        {"measure": "named combinations with more than one name", "count": int((combinations["names"] > 1).sum())},
        {"measure": "names used for more than one combination", "count": int((combinations_per_name > 1).sum())},
    ]
)
print(subset_naming.to_string(index=False))
combinations[combinations["naming"] == "partly"]

                                   measure  count
                codelist-bound value lists   2310
           ... with a subset_codelist name    296
       combinations (codelist, value_list)    254
                          ... always named     23
                          ... partly named      3
                           ... never named    228
named combinations with more than one name      0
  names used for more than one combination      0


,codelist_concept_id,value_list,variables,named,names,naming
42,C66742,N;Y,222,192,1,partly
59,C66770,cm;in;mm,39,1,1,partly
250,C99073,LEFT;RIGHT,24,8,1,partly


### Value lists with no bound codelist, by variable suffix

Where the lists with nothing to resolve to sit. `--ORRES`, `--STRESC` and
`--STRESN` are counted as result variables.

In [12]:
RESULT_SUFFIXES = ["--ORRES", "--STRESC", "--STRESN"]

no_codelist = listed[listed["scope"] == "no codelist"].merge(sheet(GRAPH, "DSS")[["ds_id", "domain"]], on="ds_id")
no_codelist["suffix"] = [
    "--" + name[len(domain):] if name.startswith(domain) else name
    for name, domain in zip(no_codelist["variable_name"], no_codelist["domain"])
]
no_codelist["result_variable"] = no_codelist["suffix"].isin(RESULT_SUFFIXES)

no_codelist_suffixes = (
    no_codelist.groupby(["suffix", "result_variable"], as_index=False)
    .agg(variables=("ds_id", "size"), dss=("ds_id", "nunique"))
    .sort_values(["variables", "suffix"], ascending=[False, True])
    .reset_index(drop=True)
)
print(
    len(no_codelist), "value lists with no bound codelist,",
    int(no_codelist["result_variable"].sum()), "on result variables",
)
no_codelist_suffixes

648 value lists with no bound codelist, 597 on result variables


,suffix,result_variable,variables,dss
0,--ORRES,True,213,213
1,--STRESC,True,213,213
2,--STRESN,True,171,171
3,--REASND,False,15,15
4,--CAT,False,7,7
5,--SCAT,False,5,5
6,--DOSTXT,False,4,4
7,--REASNE,False,4,4
8,--TRT,False,4,4
9,--DOSU,False,2,2


### Assignment states of CT-bound variables

For every variable with a bound codelist, one of three states: `assigned` (one
value fixed), `value_list` (restricted to a list), `open` (the whole codelist).
Grouped by variable suffix (variable name minus the domain prefix). Measured
only - no reading is attached to the states here.

In [13]:
dss = sheet(GRAPH, "DSS")
ct_bound = variables[variables["codelist_concept_id"].notna()].merge(dss[["ds_id", "domain"]], on="ds_id")
ct_bound["suffix"] = [
    "--" + name[len(domain):] if name.startswith(domain) else name
    for name, domain in zip(ct_bound["variable_name"], ct_bound["domain"])
]
ct_bound["state"] = "open"
ct_bound.loc[ct_bound["value_list"].notna(), "state"] = "value_list"
ct_bound.loc[
    ct_bound["assigned_term_concept_id"].notna() | ct_bound["assigned_term_value"].notna(), "state"
] = "assigned"

states = pd.crosstab(ct_bound["suffix"], ct_bound["state"]).reindex(
    columns=["assigned", "value_list", "open"], fill_value=0
)
states.insert(0, "ct_bound", states.sum(axis=1))
states["assigned_rate"] = (states["assigned"] / states["ct_bound"]).round(2)
states = states.sort_values(["ct_bound", "suffix"], ascending=[False, True]).reset_index()
print(len(ct_bound), "CT-bound variables,", len(states), "suffixes")
states.head(30)

7438 CT-bound variables, 68 suffixes


state,suffix,ct_bound,assigned,value_list,open,assigned_rate
0,--TEST,1257,1257,0,0,1.00
1,--TESTCD,1257,1257,0,0,1.00
2,--ORRESU,576,123,452,1,0.21
3,--STRESU,557,238,311,8,0.43
4,--SPEC,541,239,295,7,0.44
5,--METHOD,485,87,332,66,0.18
6,--BDAGNT,326,326,0,0,1.00
7,--STRESC,260,4,103,153,0.02
8,--ORRES,246,4,90,152,0.02
9,--CAT,218,218,0,0,1.00


### Category tokens

Same mechanism as `cosmos-graph/notebooks/50_instrument_category_resolution.ipynb`
(which stays the authority for it), applied to all tokens instead of the
instrument scope: exact `bc_short_name`, then a unique `bc_synonyms` match
(lower-cased, `;`-split). Deterministic only.

In [14]:
bc = sheet(GRAPH, "BC")
bc_categories = sheet(GRAPH, "BC_Categories")

shortname_to_bc = {}
for bc_id, short_name in zip(bc["bc_id"], bc["bc_short_name"]):
    shortname_to_bc.setdefault(str(short_name).strip(), []).append(bc_id)

synonym_to_bc = {}
for bc_id, synonyms in zip(bc["bc_id"], bc["bc_synonyms"]):
    if pd.isna(synonyms):
        continue
    for synonym in str(synonyms).split(";"):
        synonym = synonym.strip()
        if synonym:
            synonym_to_bc.setdefault(synonym.lower(), set()).add(bc_id)


def resolve(token):
    if token in shortname_to_bc:
        return "exact_name", shortname_to_bc[token]
    hits = synonym_to_bc.get(token.lower())
    if hits:
        if len(hits) == 1:
            return "synonym", sorted(hits)
        return "synonym_ambiguous", sorted(hits)
    return "unresolved", []


uses = bc_categories["category"].value_counts()
records = []
for token in sorted(uses.index):
    status, bc_ids = resolve(token)
    records.append({
        "category": token,
        "status": status,
        "bc_id": ";".join(map(str, bc_ids)),
        "uses": int(uses[token]),
    })

category_resolution = pd.DataFrame(records)
category_status = (
    category_resolution.groupby("status", as_index=False)
    .agg(tokens=("category", "size"), uses=("uses", "sum"))
)

raw_uses = int(bc["bc_categories"].dropna().str.split(";").str.len().sum())
print(raw_uses, "raw token uses in BC.bc_categories,", len(bc_categories), "distinct (BC, token) rows")
category_status

4389 raw token uses in BC.bc_categories, 4382 distinct (BC, token) rows


,status,tokens,uses
0,exact_name,125,1403
1,synonym,58,474
2,synonym_ambiguous,3,18
3,unresolved,222,2487


In [15]:
category_resolution[category_resolution["status"] == "unresolved"].sort_values(
    ["uses", "category"], ascending=[False, True]
).head(15)

,category,status,bc_id,uses
316,QRS,unresolved,,290
387,Trial Summary,unresolved,,192
231,Laboratory Tests,unresolved,,186
339,Respiratory System Findings,unresolved,,152
3,ADAS-Cog,unresolved,,112
5,ADC,unresolved,,112
336,Reproductive Findings,unresolved,,99
81,Chemistry Tests,unresolved,,90
355,Subject Characteristics,unresolved,,76
96,Clinical Trial Attribute,unresolved,,66


### Scope assertion against notebook 50

The resolution above is a copy of the mechanism in
`50_instrument_category_resolution.ipynb`. Restricted to the tokens on BCs
with a DSS in QS/FT/RS it must reproduce the counts that notebook wrote to
`Instrument_Category_Resolution.xlsx`. Stops if the copy has drifted.

In [16]:
INSTR_DOMAINS = {"QS", "FT", "RS"}
NOTEBOOK_50 = REPO / "cosmos-graph" / "reports" / "Instrument_Category_Resolution.xlsx"

dss_domains = sheet(GRAPH, "DSS")
instrument_bc_ids = set(dss_domains.loc[dss_domains["domain"].isin(INSTR_DOMAINS), "bc_id"])
instrument_tokens = set(bc_categories.loc[bc_categories["bc_id"].isin(instrument_bc_ids), "category"].dropna())
instrument_status = category_resolution[category_resolution["category"].isin(instrument_tokens)]["status"].value_counts()

notebook_50 = pd.read_excel(NOTEBOOK_50, sheet_name="Summary")
expected = {status: int(count) for status, count in zip(notebook_50["status"], notebook_50["token_count"])}
observed = {status: int(instrument_status.get(status, 0)) for status in expected}
if observed != expected:
    raise ValueError(f"instrument-scope resolution differs from notebook 50: {observed} != {expected}")
print(len(instrument_tokens), "instrument-scope tokens:", observed, "- same as notebook 50")

106 instrument-scope tokens: {'exact_name': 41, 'synonym': 39, 'synonym_ambiguous': 0, 'unresolved': 26} - same as notebook 50


### Integrity checks already in the graph

`Unresolved` = concept IDs not found in the CT package. `Anomalies` =
pinned term not in the bound codelist. Both are membership checks, i.e.
validation of published content against published content.

In [17]:
checks = pd.DataFrame(
    [
        {"check": "Unresolved concept IDs", "rows": len(sheet(GRAPH_CT, "Unresolved"))},
        {"check": "Anomalies (pinned term not in bound codelist)", "rows": len(sheet(GRAPH_CT, "Anomalies"))},
    ]
)
checks

,check,rows
0,Unresolved concept IDs,5
1,Anomalies (pinned term not in bound codelist),0


## Key measures

Long format (measure, value), the diffable part of this notebook. A change
between packages is a finding, not a failure - see the final cell.

In [18]:
def measures(df, prefix, label_cols, value_col):
    labels = df[label_cols].astype(str).agg(" / ".join, axis=1)
    return pd.DataFrame({"measure": prefix + ": " + labels, "value": df[value_col].astype(int)})


state_totals = states[["assigned", "value_list", "open"]].sum().rename_axis("state").reset_index(name="variables")

key_measures = pd.concat(
    [
        measures(audit, "rows", ["link_kind", "provenance", "statement"], "rows"),
        measures(summary, "rows per link kind", ["link_kind", "provenance"], "rows"),
        measures(grain, "LOINC", ["measure"], "count"),
        measures(value_list_detail, "value lists: variables", ["scope"], "variables"),
        measures(subset_naming, "subset_codelist naming", ["measure"], "count"),
        measures(no_codelist_suffixes[no_codelist_suffixes["result_variable"]], "no-codelist value lists on result variables", ["suffix"], "variables"),
        measures(state_totals, "assignment state", ["state"], "variables"),
        measures(category_status, "category tokens", ["status"], "tokens"),
        measures(category_status, "category uses", ["status"], "uses"),
        measures(checks, "integrity", ["check"], "rows"),
    ],
    ignore_index=True,
)
if key_measures["measure"].duplicated().any():
    raise ValueError("duplicate measure names: " + ", ".join(key_measures.loc[key_measures["measure"].duplicated(), "measure"]))
print(len(key_measures), "key measures")
key_measures

67 key measures


,measure,value
0,rows: identity / published / BC is an NCIt con...,1475
1,rows: identity / published / DEC is an NCIt co...,6029
2,rows: identity / published / TESTCD is an NCIt...,5988
3,rows: identity / published / codelist is an NC...,297
4,rows: membership / published / concept is a pe...,17742
...,...,...
62,category uses: synonym,474
63,category uses: synonym_ambiguous,18
64,category uses: unresolved,2487
65,integrity: Unresolved concept IDs,5


## Write the audit workbook

In [19]:
with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    audit.to_excel(writer, sheet_name="Link_Kinds", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)
    match_methods.to_excel(writer, sheet_name="Instrument_Match_Method", index=False)
    external_systems.to_excel(writer, sheet_name="External_Systems", index=False)
    grain.to_excel(writer, sheet_name="LOINC_Grains", index=False)
    pins.to_excel(writer, sheet_name="Pins", index=False)
    value_list_detail.to_excel(writer, sheet_name="Value_List_Detail", index=False)
    subset_naming.to_excel(writer, sheet_name="Subset_Naming", index=False)
    combinations.to_excel(writer, sheet_name="Subset_Naming_Combinations", index=False)
    no_codelist_suffixes.to_excel(writer, sheet_name="No_Codelist_Suffixes", index=False)
    states.to_excel(writer, sheet_name="Assignment_States", index=False)
    category_status.to_excel(writer, sheet_name="Category_Status", index=False)
    category_resolution.to_excel(writer, sheet_name="Category_Resolution", index=False)
    checks.to_excel(writer, sheet_name="Integrity_Checks", index=False)
    key_measures.to_excel(writer, sheet_name="Key_Measures", index=False)

print("written:", OUT)

written: /sessions/rcw-011ndgbpomxbp7qzdpv1ucjv/mnt/cdisc-for-ai/link-semantics/interim/Link_Kind_Audit.xlsx


## How to read this

- `identity`, `membership` and `binding` rows are **not** mappings. In RDF they
  need a predicate that is not transitive, so that merging graphs cannot create
  equivalences that no source asserts.
- `denotation` rows link a model element to a concept. `skos:exactMatch` treats
  the two as peers and is transitive; an annotation property does not.
- `pinned_value` rows are values in a fixed context, not statements about
  concept equivalence. The context is the DSS.
- `value_list` rows restrict a variable to a set of values in a fixed context.
  They sit between `binding` (the whole codelist) and `pinned_value` (one
  value). A value list and an assigned value never occur on the same variable.
- `category_tag` rows have a label at the object end, not an identifier. The
  same holds for the `subset_codelist` names and for value lists with no bound
  codelist. The category resolution status says which tokens can be tied to a
  BC by a deterministic join; it is derived, not published.
- `mapping` rows are the only ones where a justification is meaningful. Two of
  them already carry one: `Instrument_Match_Method` and the cross-reference
  `rationale`.

Next: the code-collision check - every C-code used in more than one role, which
is where a transitive predicate would bite in a merged graph.

## Version-bump diff (optional)

Set `PRIOR_FILE` (cell 2) to a previous `Link_Kind_Audit.xlsx` to compare its
`Key_Measures` against this run. No-ops if unset or missing.

In [20]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    prior = pd.read_excel(PRIOR_FILE, sheet_name="Key_Measures")
    prev = dict(zip(prior["measure"], prior["value"]))
    curr = dict(zip(key_measures["measure"], key_measures["value"]))
    added = sorted(set(curr) - set(prev))
    dropped = sorted(set(prev) - set(curr))
    changed = [(m, prev[m], curr[m]) for m in sorted(set(prev) & set(curr)) if prev[m] != curr[m]]
    print(f"Prior:    {PRIOR_FILE}")
    print(f"Measures: prior {len(prev)} -> current {len(curr)}")
    print(f"Added measures:   {added}")
    print(f"Dropped measures: {dropped}")
    print("Value changes (measure: prior -> current):")
    for m, a, b in changed:
        print(f"  {m}: {a} -> {b}")
    if not changed:
        print("  (none)")
else:
    print("PRIOR_FILE not set or missing - skipping version diff.")

PRIOR_FILE not set or missing - skipping version diff.
